**Importing Libraries**

In [1]:
import pandas as pd
import hashlib
from datetime import datetime
from sqlalchemy import create_engine

**Connecting to Supabase**

In [2]:
from google.colab import userdata

# Load values from Colab Secrets
db_password = userdata.get('SUPABASE_PASSWORD')
project_ref = userdata.get('SUPABASE_PROJECT_REF')
db_host = userdata.get('SUPABASE_HOST')

# Database settings
db_port = 5432

# Create connection engine
engine = create_engine(
    "postgresql+psycopg2://",
    connect_args={
        "user": f"postgres.{project_ref}",
        "password": db_password,
        "host": db_host,
        "port": db_port,
        "database": "postgres",
        "sslmode": "require"
    }
)

print("Connection engine built successfully!")

Connection engine built successfully!


# **Bronze Layer**

In [ ]:
import pandas as pd
import hashlib
from datetime import datetime, timezone

# 1. Read file exactly as it is
try:
    df = pd.read_csv('train.csv')
    print("Raw file loaded successfully!")
except FileNotFoundError:
    raise FileNotFoundError("Could not find 'train.csv'. Please upload it to your Colab files panel on the left sidebar!")


Raw file loaded successfully!


In [ ]:
# 2. Setup lineage audit constants using modern timezone-aware UTC datetime
CURRENT_UTC_TIME = datetime.now(timezone.utc)
BATCH_ID = f"BATCH_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
SOURCE_FILE = "train.csv"
DATA_SOURCE = "Colab_Raw_Bronze_Pipeline"

print("Piping uncleaned data to Supabase...")

Piping uncleaned data to Supabase...


In [ ]:
# --- 4. RAW CUSTOMERS ---
customers_df = df[['Customer ID', 'Customer Name', 'Segment']].copy().astype(str)
customers_df['batch_id'] = BATCH_ID
customers_df['source_file_name'] = SOURCE_FILE
customers_df['data_source'] = DATA_SOURCE
customers_df['is_active'] = True
customers_df['load_date'] = CURRENT_UTC_TIME
customers_df['record_created_date'] = CURRENT_UTC_TIME
customers_df['record_updated_date'] = CURRENT_UTC_TIME

def cust_full_hash(row):
    combined = f"{row['Customer ID']}{row['Customer Name']}{row['Segment']}"
    return hashlib.md5(combined.encode('utf-8')).hexdigest()
customers_df['row_hash'] = customers_df.apply(cust_full_hash, axis=1)

customers_df.to_sql('bronze_customers', con=engine, if_exists='append', index=False)
print("Loaded uncleaned public.bronze_customers")

Loaded uncleaned public.bronze_customers


In [ ]:
# --- 4. RAW PRODUCTS ---
products_df = df[['Product ID', 'Category', 'Sub-Category', 'Product Name']].copy()
products_df['batch_id'] = BATCH_ID
products_df['source_file_name'] = SOURCE_FILE
products_df['data_source'] = DATA_SOURCE
products_df['is_active'] = True
products_df['load_date'] = CURRENT_UTC_TIME
products_df['record_created_date'] = CURRENT_UTC_TIME
products_df['record_updated_date'] = CURRENT_UTC_TIME

def prod_full_hash(row):
    combined = f"{row['Product ID']}{row['Category']}{row['Sub-Category']}{row['Product Name']}"
    return hashlib.md5(combined.encode('utf-8')).hexdigest()
products_df['row_hash'] = products_df.apply(prod_full_hash, axis=1)

products_df.to_sql('bronze_products', con=engine, if_exists='append', index=False)
print("Loaded uncleaned public.bronze_products")

Loaded uncleaned public.bronze_products


In [ ]:
# --- 5. RAW ORDERS ---
order_cols = ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Product ID', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Sales']
# Force data types to string to completely safeguard raw uncleaned formatting from parsing errors
orders_df = df[order_cols].copy().astype(str)
orders_df['batch_id'] = BATCH_ID
orders_df['source_file_name'] = SOURCE_FILE
orders_df['data_source'] = DATA_SOURCE
orders_df['is_active'] = True
orders_df['load_date'] = CURRENT_UTC_TIME
orders_df['record_created_date'] = CURRENT_UTC_TIME
orders_df['record_updated_date'] = CURRENT_UTC_TIME

def ord_full_hash(row):
    combined = "".join([row[col] for col in order_cols])
    return hashlib.md5(combined.encode('utf-8')).hexdigest()
orders_df['row_hash'] = orders_df.apply(ord_full_hash, axis=1)

orders_df.to_sql('bronze_orders', con=engine, if_exists='append', index=False)
print("Loaded uncleaned public.bronze_orders")

Loaded uncleaned public.bronze_orders


In [ ]:
print("Done! Your uncleaned source data is now fully preserved in your Bronze database layer.")

Done! Your uncleaned source data is now fully preserved in your Bronze database layer.


In [ ]:
from sqlalchemy import text

print("Checking live database row counts...")

with engine.connect() as conn:
    # 1. Check Customers row count
    cust_count = conn.execute(text('SELECT COUNT(*) FROM public.bronze_customers;')).scalar()
    print(f"Row count in bronze_customers: {cust_count}")

    # 2. Check Products row count
    prod_count = conn.execute(text('SELECT COUNT(*) FROM public.bronze_products;')).scalar()
    print(f"Row count in bronze_products:  {prod_count}")

    # 3. Check Orders row count
    ord_count = conn.execute(text('SELECT COUNT(*) FROM public.bronze_orders;')).scalar()
    print(f"Row count in bronze_orders:    {ord_count}")

Checking live database row counts...
Row count in bronze_customers: 9800
Row count in bronze_products:  9800
Row count in bronze_orders:    9800


# **Silver Layer**

In [ ]:
import pandas as pd
import numpy as np

**Data Cleaning**

In [ ]:
# --- 1. CUSTOMERS ---
df_bronze_customers = pd.read_sql_table('bronze_customers', schema='bronze', con=engine)

# Rename to clean snake_case mapping
df_cleaned_customers = df_bronze_customers.rename(columns={
    'Customer ID': 'customer_id',
    'Customer Name': 'customer_name',
    'Segment': 'segment'
})[['customer_id', 'customer_name', 'segment']]

# Activity: Remove duplicate records based on the new primary key name
df_cleaned_customers = df_cleaned_customers.drop_duplicates(subset=['customer_id'], keep='last').copy()

# Activity: Handle null or missing values
df_cleaned_customers['customer_name'] = df_cleaned_customers['customer_name'].fillna('Unknown').str.strip()
df_cleaned_customers['segment'] = df_cleaned_customers['segment'].fillna('Consumer').str.strip()

In [ ]:
# --- 2. PRODUCTS ---
df_bronze_products = pd.read_sql_table('bronze_products', schema='bronze', con=engine)

# Rename to clean snake_case mapping
df_cleaned_products = df_bronze_products.rename(columns={
    'Product ID': 'product_id',
    'Category': 'category',
    'Sub-Category': 'sub_category',
    'Product Name': 'product_name'
})[['product_id', 'category', 'sub_category', 'product_name']]

# Activity: Remove duplicate records based on product_id
df_cleaned_products = df_cleaned_products.drop_duplicates(subset=['product_id'], keep='last').copy()

# Activity: Handle null or missing values
df_cleaned_products['category'] = df_cleaned_products['category'].fillna('Uncategorized').str.strip()
df_cleaned_products['sub_category'] = df_cleaned_products['sub_category'].fillna('Uncategorized').str.strip()
df_cleaned_products['product_name'] = df_cleaned_products['product_name'].fillna('Unknown Product').str.strip()

In [ ]:
# --- 2. ORDERS ---
df_bronze_orders = pd.read_sql_table('bronze_orders', schema='bronze', con=engine)

# Rename ALL columns to clean snake_case mapping
df_cleaned_orders = df_bronze_orders.rename(columns={
    'Row ID': 'row_id',
    'Order ID': 'order_id',
    'Order Date': 'order_date',
    'Ship Date': 'ship_date',
    'Ship Mode': 'ship_mode',
    'Customer ID': 'customer_id',
    'Product ID': 'product_id',
    'Country': 'country',
    'City': 'city',
    'State': 'state',
    'Postal Code': 'postal_code',
    'Region': 'region',
    'Sales': 'sales'
})

# Activity: Remove duplicate transactional rows
df_cleaned_orders = df_cleaned_orders.drop_duplicates(subset=['row_id', 'order_id'], keep='last').copy()

# Activity: Standardize date formats
df_cleaned_orders['order_date'] = pd.to_datetime(df_cleaned_orders['order_date'], dayfirst = True, errors='coerce')
df_cleaned_orders['ship_date'] = pd.to_datetime(df_cleaned_orders['ship_date'], dayfirst = True, errors='coerce')

# Activity: Standardize state names and regional data to uppercase
df_cleaned_orders['state'] = df_cleaned_orders['state'].str.strip().str.upper()
df_cleaned_orders['region'] = df_cleaned_orders['region'].str.strip().str.upper()
df_cleaned_orders['country'] = df_cleaned_orders['country'].str.strip().str.upper()
df_cleaned_orders['city'] = df_cleaned_orders['city'].str.strip()

# Activity: Validate sales values (convert messy numbers/strings to clean decimal floats)
df_cleaned_orders['sales'] = pd.to_numeric(df_cleaned_orders['sales'], errors='coerce')

**Data Quality Checks**

In [ ]:
# Customers Primary Key
df_silver_customers_prepped = df_cleaned_customers[
    df_cleaned_customers['customer_id'].notna() & (df_cleaned_customers['customer_id'] != '')
]

In [ ]:
# Products Primary Key
df_silver_products_prepped = df_cleaned_products[
    df_cleaned_products['product_id'].notna() & (df_cleaned_products['product_id'] != '')
]

In [ ]:
# Validating Orders
df_silver_orders_prepped = df_cleaned_orders[
    # Key Integrity Checks (IDs/Keys must exist)
    df_cleaned_orders['order_id'].notna() &
    df_cleaned_orders['customer_id'].notna() &
    df_cleaned_orders['product_id'].notna() &

    # Business Logic Rules
    (df_cleaned_orders['sales'] > 0) &
    df_cleaned_orders['order_date'].notna() &

    # State Validation check
    (df_cleaned_orders['state'].str.len() >= 2)
].copy()

**Creating Silver Tables**

In [ ]:
# 1. SCD TYPE 2 FOR CUSTOMERS
try:
    df_existing_cust = pd.read_sql_query("SELECT * FROM silver.silver_customers WHERE is_active = true", con=engine)
except Exception:
    df_existing_cust = pd.DataFrame(columns=['customer_id', 'customer_name', 'segment'])

# Merge incoming data against current live active data
merged_cust = df_silver_customers_prepped.merge(df_existing_cust, on='customer_id', how='left', suffixes=('_new', '_old'))

# Condition A: Completely brand new customer entries
new_cust = merged_cust[merged_cust['is_active'].isna()][['customer_id', 'customer_name_new', 'segment_new']].rename(
    columns={'customer_name_new': 'customer_name', 'segment_new': 'segment'}
)

# Condition B: Existing customer records where attributes have changed
changed_cust = merged_cust[
    (merged_cust['is_active'] == True) &
    ((merged_cust['customer_name_new'] != merged_cust['customer_name_old']) |
     (merged_cust['segment_new'] != merged_cust['segment_old']))
]

# Deactivate old versions of changed rows in Supabase
if not changed_cust.empty:
    with engine.connect() as conn:
        for _, row in changed_cust.iterrows():
            conn.execute(text("""
                UPDATE public.silver_customers
                SET is_active = false, record_updated_date = CURRENT_TIMESTAMP
                WHERE customer_id = :cust_id AND is_active = true;
            """), {"cust_id": row['customer_id']})
    print(f"-> Historical tracking: Deactivated {len(changed_cust)} expired customer profiles.")

# Append new entries + fresh versions of updated records
cust_to_insert = pd.concat([new_cust, changed_cust[['customer_id', 'customer_name_new', 'segment_new']].rename(
    columns={'customer_name_new': 'customer_name', 'segment_new': 'segment'}
)])

if not cust_to_insert.empty:
    cust_to_insert['is_active'] = True
    cust_to_insert.to_sql('silver_customers', con=engine, if_exists='append', index=False)
    print(f"Inserted {len(cust_to_insert)} new historical records into 'silver_customers'.")


In [ ]:
# 2. SCD TYPE 2 FOR PRODUCTS
try:
    df_existing_prod = pd.read_sql_query("SELECT * FROM silver.silver_products WHERE is_active = true", con=engine)
except Exception:
    df_existing_prod = pd.DataFrame(columns=['product_id', 'category', 'sub_category', 'product_name'])

merged_prod = df_silver_products_prepped.merge(df_existing_prod, on='product_id', how='left', suffixes=('_new', '_old'))

new_prod = merged_prod[merged_prod['is_active'].isna()][['product_id', 'category_new', 'sub_category_new', 'product_name_new']].rename(
    columns={'category_new': 'category', 'sub_category_new': 'sub_category', 'product_name_new': 'product_name'}
)

changed_prod = merged_prod[
    (merged_prod['is_active'] == True) &
    ((merged_prod['category_new'] != merged_prod['category_old']) |
     (merged_prod['sub_category_new'] != merged_prod['sub_category_old']) |
     (merged_prod['product_name_new'] != merged_prod['product_name_old']))
]

if not changed_prod.empty:
    with engine.connect() as conn:
        for _, row in changed_prod.iterrows():
            conn.execute(text("""
                UPDATE public.silver_products
                SET is_active = false, record_updated_date = CURRENT_TIMESTAMP
                WHERE product_id = :prod_id AND is_active = true;
            """), {"prod_id": row['product_id']})
    print(f"-> Historical tracking: Deactivated {len(changed_prod)} expired product SKUs.")

prod_to_insert = pd.concat([new_prod, changed_prod[['product_id', 'category_new', 'sub_category_new', 'product_name_new']].rename(
    columns={'category_new': 'category', 'sub_category_new': 'sub_category', 'product_name_new': 'product_name'}
)])

if not prod_to_insert.empty:
    prod_to_insert['is_active'] = True
    prod_to_insert.to_sql('silver_products', con=engine, if_exists='append', index=False)
    print(f"Inserted {len(prod_to_insert)} new historical records into 'silver_products'.")

In [ ]:
# 3. ORDERS

missing_dates = df_silver_orders_prepped['order_date'].isna().sum()

if missing_dates > 0:
    df_silver_orders_prepped = df_silver_orders_prepped.dropna(
        subset=['order_date']
    )
    print(f"{missing_dates} rows removed because order_date was missing.")
else:
    print("All rows have valid order_date.")

All rows have valid order_date.


In [ ]:
# Display Number of Records in Silver Table

from sqlalchemy import text

with engine.connect() as conn:

    customers_count = conn.execute(
        text("SELECT COUNT(*) FROM silver.silver_customers")
    ).scalar()

    products_count = conn.execute(
        text("SELECT COUNT(*) FROM silver.silver_products")
    ).scalar()

    orders_count = conn.execute(
        text("SELECT COUNT(*) FROM silver.silver_orders")
    ).scalar()


print(f"silver_customers: {customers_count} rows")
print(f"silver_products:  {products_count} rows")
print(f"silver_orders:    {orders_count} rows")


silver_customers: 793 rows
silver_products:  1861 rows
silver_orders:    9800 rows


# **Gold Layer**

In [ ]:
import pandas as pd

# Orders
df_orders = pd.read_sql(
    "SELECT * FROM silver_orders",
    con=engine
)

# Products
df_products = pd.read_sql(
    "SELECT * FROM silver_products",
    con=engine
)

# Customers (ONLY ACTIVE RECORDS FOR SCD TYPE 2)
df_customers = pd.read_sql(
    """
    SELECT *
    FROM silver_customers
    WHERE is_active = true
    """,
    con=engine
)

print("Silver tables loaded successfully.")

Silver tables loaded successfully.


**Sales Summary**

In [ ]:
def create_sales_summary():

    total_sales = df_orders['sales'].sum()

    total_orders = df_orders['order_id'].nunique()

    avg_order_value = round(
        total_sales / total_orders,
        2
    )

    gold_sales_summary = pd.DataFrame({
        'total_sales': [total_sales],
        'total_orders': [total_orders],
        'avg_order_value': [avg_order_value]
    })

    gold_sales_summary.to_sql(
        'gold_sales_summary',
        con=engine,
        if_exists='replace',
        index=False
    )

    print("gold_sales_summary created")

**Product Performance**

In [ ]:
def create_product_performance():

    product_df = df_orders.merge(
        df_products,
        on='product_id',
        how='left'
    )

    gold_product_performance = (
        product_df
        .groupby(
            [
                'product_id',
                'product_name',
                'category',
                'sub_category'
            ]
        )
        .agg(
            product_revenue=('sales', 'sum'),
            total_orders=('order_id', 'nunique')
        )
        .reset_index()
        .sort_values(
            by='product_revenue',
            ascending=False
        )
    )

    gold_product_performance.to_sql(
        'gold_product_performance',
        con=engine,
        if_exists='replace',
        index=False
    )

    print("gold_product_performance created")

**Customer Analysis**

In [ ]:
def create_customer_analysis():

    customer_df = df_orders.merge(
        df_customers,
        on='customer_id',
        how='left'
    )

    gold_customer_analysis = (
        customer_df
        .groupby(
            [
                'customer_id',
                'customer_name',
                'segment'
            ]
        )
        .agg(
            customer_revenue=('sales', 'sum'),
            order_frequency=('order_id', 'nunique')
        )
        .reset_index()
        .sort_values(
            by='customer_revenue',
            ascending=False
        )
    )

    gold_customer_analysis.to_sql(
        'gold_customer_analysis',
        con=engine,
        if_exists='replace',
        index=False
    )

    print("gold_customer_analysis created")

**Region Analysis**

In [ ]:
def create_region_analysis():

    gold_region_analysis = (
        df_orders
        .groupby(
            [
                'region',
                'state',
                'city'
            ]
        )
        .agg(
            total_sales=('sales', 'sum'),
            total_orders=('order_id', 'nunique')
        )
        .reset_index()
    )

    gold_region_analysis.to_sql(
        'gold_region_analysis',
        con=engine,
        if_exists='replace',
        index=False
    )

    print("gold_region_analysis created")

**ETL Pipeline**

In [ ]:
def run_gold_etl():

    create_sales_summary()

    create_product_performance()

    create_customer_analysis()

    create_region_analysis()

    print("Gold Layer ETL Completed Successfully")

In [ ]:
run_gold_etl()

gold_sales_summary created
gold_product_performance created
gold_customer_analysis created
gold_region_analysis created
Gold Layer ETL Completed Successfully
